# 첫 번째 Trace 만들기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `@traceable` 데코레이터로 일반 Python 함수를 LangSmith에 추적할 수 있어요
2. `run_type`, `name` 파라미터로 Trace를 분류하고 이름을 지정할 수 있어요
3. 중첩된 `@traceable` 함수로 부모-자식 Run 계층 구조(Run Tree)를 만들 수 있어요
4. `ls.trace()` 컨텍스트 매니저와 `wrap_openai()`를 사용해 추적할 수 있어요

## 사전 지식

- `01-Setup-and-Overview.ipynb` 완료 (LangSmith 계정, API 키, 환경 변수 설정)
- Python 데코레이터(`@`) 기본 개념
- Python 컨텍스트 매니저(`with` 구문) 기본 개념

## 추적(Tracing)의 다섯 가지 방법

LangSmith에는 코드를 추적하는 방법이 다섯 가지 있어요. 상황에 따라 적합한 방법을 선택해요.

```mermaid
flowchart TD
    A[추적 방법 선택]:::process --> B{어떤 코드를 추적하나요?}
    B --> C[LangChain 기반 코드]:::input
    B --> D[일반 Python 함수]:::input
    B --> E[OpenAI SDK 직접 사용]:::input
    C --> F[방법 1: 환경변수만 설정<br/>LANGSMITH_TRACING=true]:::output
    D --> G[방법 2: @traceable 데코레이터]:::output
    D --> H[방법 3: ls.trace() 컨텍스트 매니저]:::output
    E --> I[방법 4: wrap_openai() 래핑]:::output
    F --> J[LangSmith UI<br/>Trace 확인]:::storage
    G --> J
    H --> J
    I --> J

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

| 방법 | 사용 상황 | 코드 변경 |
|------|----------|----------|
| **환경변수만 설정** | LangChain 코드 전체 | 없음 |
| **@traceable** | 특정 Python 함수를 추적 | 데코레이터 추가 |
| **ls.trace()** | 코드 블록 단위로 추적 | with 블록 추가 |
| **wrap_openai()** | 순수 OpenAI SDK 사용 | 클라이언트 래핑 |
| **RunTree API** | 세밀한 수동 제어 (고급) | 많은 코드 추가 |

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
from dotenv import load_dotenv
import os

# .env 파일에서 LANGSMITH_TRACING, LANGSMITH_API_KEY 등을 로드해요
load_dotenv()

# 이 노트북 전용 프로젝트
os.environ["LANGSMITH_PROJECT"] = "Ch02-First-Trace"

print(f"LANGSMITH_TRACING: {os.getenv('LANGSMITH_TRACING', '')}")
print(f"프로젝트: {os.environ['LANGSMITH_PROJECT']}")

## 1. @traceable 데코레이터 기본 사용법

`@traceable` 데코레이터는 LangChain과 무관한 일반 Python 함수를 LangSmith에 추적하는 가장 간단한 방법이에요. 함수 정의 위에 `@traceable`을 붙이기만 하면 돼요.

> 🔑 **핵심 개념**: `@traceable`은 함수의 입력과 출력을 자동으로 캡처해요. 함수가 실행되면 LangSmith에 새 Run이 생성되고, 함수 이름이 Run 이름으로 사용돼요.

> 💡 **실무 팁**: `run_type`을 올바르게 설정하면 LangSmith UI에서 Run 종류별로 색깔이 다르게 표시돼요. 특히 `run_type="llm"`으로 설정하면 토큰 수와 비용 정보가 자동으로 렌더링돼요.

In [2]:
# ---------------------------------------------------
# @traceable 데코레이터 기본 사용법
# ---------------------------------------------------
from langsmith import traceable

# @traceable만 붙이면 함수 이름이 Run 이름으로 사용돼요
@traceable
def simple_greeting(name: str) -> str:
    """간단한 인사 함수예요"""
    return f"안녕하세요, {name}님!"

# @traceable로 파라미터를 지정해 run_type과 name을 설정할 수 있어요
@traceable(
    run_type="chain",          # Run 유형: llm, chain, tool, retriever, embedding
    name="인사 생성 파이프라인"  # UI에 표시되는 이름 (기본값: 함수 이름)
)
def greeting_pipeline(name: str, language: str = "한국어") -> dict:
    """인사 메시지를 생성하는 파이프라인이에요"""
    # 이 함수 내부의 모든 작업이 하나의 Run으로 추적돼요
    greeting = simple_greeting(name)  # 자식 Run이 생성돼요
    return {
        "greeting": greeting,
        "language": language,
        "length": len(greeting)
    }

# 함수 호출 — LangSmith에 자동으로 Trace가 기록돼요
result = greeting_pipeline("김철수", language="한국어")
print(f"결과: {result}")
print("\nLangSmith UI에서 '인사 생성 파이프라인' Run을 확인해보세요!")

결과: {'greeting': '안녕하세요, 김철수님!', 'language': '한국어', 'length': 12}

LangSmith UI에서 '인사 생성 파이프라인' Run을 확인해보세요!


### @traceable 파라미터 정리

| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `run_type` | str | Run의 유형을 지정해요 (기본값: `"chain"`) |
| `name` | str | UI에 표시될 이름이에요 (기본값: 함수 이름) |
| `tags` | list[str] | 검색·필터링용 태그예요 |
| `metadata` | dict | 추가 컨텍스트 정보예요 |
| `project_name` | str | 기본 프로젝트 대신 특정 프로젝트에 기록해요 |

## 2. 중첩 @traceable로 Run Tree 만들기

`@traceable` 함수 안에서 다른 `@traceable` 함수를 호출하면, 부모-자식 Run 계층 구조가 자동으로 만들어져요. LangSmith UI의 **Run Tree**에서 이 계층 구조를 시각적으로 확인할 수 있어요.

> 🎯 **강의 포인트**: 실무에서 RAG 파이프라인을 추적할 때 이 계층 구조가 매우 중요해요. 전체 체인 → 검색 → LLM 호출 → 파싱 과정의 각 단계에서 얼마나 걸렸는지, 입출력이 어떻게 됐는지 한눈에 볼 수 있어요.

In [3]:
# ---------------------------------------------------
# 중첩 @traceable로 Run Tree 구성하기
# ---------------------------------------------------
from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# LLM 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1단계: 도구 함수 — run_type="tool"로 지정해요
@traceable(run_type="tool", name="컨텍스트 검색")
def retrieve_context(question: str) -> str:
    """질문과 관련된 컨텍스트를 검색해요 (예시: 하드코딩)"""
    # 실제 서비스에서는 여기서 VectorStore를 조회해요
    context_db = {
        "langsmith": "LangSmith는 LangChain이 만든 LLM 관측성 플랫폼이에요.",
        "langchain": "LangChain은 LLM 애플리케이션 개발 프레임워크예요.",
        "langgraph": "LangGraph는 복잡한 에이전트 워크플로우를 구축하는 라이브러리예요.",
    }
    # 질문에서 키워드를 추출해 컨텍스트를 반환해요
    for keyword, context in context_db.items():
        if keyword in question.lower():
            return context
    return "관련 정보를 찾을 수 없어요."

# 2단계: LLM 호출 함수 — run_type="llm"으로 지정해요
@traceable(run_type="llm", name="LLM 응답 생성")
def generate_answer(question: str, context: str) -> str:
    """컨텍스트를 바탕으로 LLM이 답변을 생성해요"""
    messages = [
        SystemMessage("주어진 컨텍스트만을 사용해 질문에 답하세요. 한 문장으로 답해요."),
        HumanMessage(f"컨텍스트: {context}\n\n질문: {question}")
    ]
    response = model.invoke(messages)
    return response.content

# 3단계: 전체 파이프라인 — 부모 Run이 되어 위 두 함수를 자식 Run으로 포함해요
@traceable(run_type="chain", name="QA 파이프라인")
def qa_pipeline(question: str) -> dict:
    """질문에 대한 답변을 생성하는 전체 파이프라인이에요"""
    # 각 단계가 자식 Run으로 추적돼요
    context = retrieve_context(question)   # 자식 Run 1: 컨텍스트 검색
    answer = generate_answer(question, context)  # 자식 Run 2: LLM 호출
    return {
        "question": question,
        "context": context,
        "answer": answer
    }

# 파이프라인 실행
result = qa_pipeline("LangSmith가 무엇인가요?")
print(f"질문: {result['question']}")
print(f"컨텍스트: {result['context']}")
print(f"답변: {result['answer']}")
print("\nLangSmith UI에서 'QA 파이프라인' Run의 하위에 두 개의 자식 Run을 확인해보세요!")

질문: LangSmith가 무엇인가요?
컨텍스트: LangSmith는 LangChain이 만든 LLM 관측성 플랫폼이에요.
답변: LangSmith는 LangChain이 만든 LLM 관측성 플랫폼입니다.

LangSmith UI에서 'QA 파이프라인' Run의 하위에 두 개의 자식 Run을 확인해보세요!


LangSmith UI에서 이 Trace를 보면 다음과 같은 Run Tree를 볼 수 있어요:

```
QA 파이프라인 (chain)          ← 부모 Run
├── 컨텍스트 검색 (tool)       ← 자식 Run 1
└── LLM 응답 생성 (llm)        ← 자식 Run 2
    └── ChatOpenAI (llm)       ← 자동 추적된 LLM 호출
```

각 Run에서 확인할 수 있는 정보:
- **입력값**: 함수에 전달된 인자
- **출력값**: 함수의 반환 값
- **소요 시간**: 각 단계의 실행 시간
- **토큰 수/비용**: LLM Run에서는 사용된 토큰과 예상 비용도 확인 가능해요

## 3. langsmith_extra로 런타임 파라미터 전달

`@traceable` 함수를 호출할 때 `langsmith_extra` 파라미터를 전달하면, 해당 호출에만 적용되는 태그나 메타데이터를 지정할 수 있어요. 함수 정의를 수정하지 않아도 돼요.

> 💡 **실무 팁**: `langsmith_extra`는 같은 함수를 다른 환경(개발/운영)에서 실행할 때 환경 정보를 동적으로 추가하기 좋아요. 함수 코드는 그대로 두고 호출하는 쪽에서만 메타데이터를 붙이면 돼요.

In [4]:
# ---------------------------------------------------
# langsmith_extra로 런타임 파라미터 전달하기
# ---------------------------------------------------
# ============================================================
# TODO: langsmith_extra의 tags와 metadata를 원하는 값으로 바꿔보세요
# 힌트: tags는 리스트, metadata는 딕셔너리 형태예요
# 예상 결과: LangSmith UI에서 설정한 태그로 Run을 검색할 수 있어요
# ============================================================

@traceable(run_type="chain", name="문서 요약기")
def summarize_document(document: str) -> str:
    """문서를 요약해요 (예시 함수)"""
    # 실제로는 LLM을 호출하지만 여기서는 예시로 슬라이싱해요
    return document[:50] + "... (요약됨)"

# 같은 함수를 다른 컨텍스트로 호출할 때 langsmith_extra를 사용해요
result1 = summarize_document(
    "LangSmith는 LLM 애플리케이션 개발을 위한 강력한 플랫폼입니다.",
    langsmith_extra={
        "tags": ["production", "user-request"],   # 이 호출에만 적용될 태그
        "metadata": {
            "user_id": "user-001",
            "session_id": "sess-abc123",
            "environment": "production"
        }
    }
)
print(f"운영 환경 결과: {result1}")

# 개발 환경에서 같은 함수 호출 — 다른 태그와 메타데이터 사용
result2 = summarize_document(
    "Python은 배우기 쉬운 프로그래밍 언어입니다.",
    langsmith_extra={
        "tags": ["development", "test"],
        "metadata": {"environment": "development", "tester": "QA팀"}
    }
)
print(f"개발 환경 결과: {result2}")
print("\nLangSmith UI에서 'production'과 'development' 태그로 필터링해보세요!")

운영 환경 결과: LangSmith는 LLM 애플리케이션 개발을 위한 강력한 플랫폼입니다.... (요약됨)
개발 환경 결과: Python은 배우기 쉬운 프로그래밍 언어입니다.... (요약됨)

LangSmith UI에서 'production'과 'development' 태그로 필터링해보세요!


## 4. ls.trace() 컨텍스트 매니저

`ls.trace()`는 `with` 블록 단위로 추적 범위를 지정하는 컨텍스트 매니저예요. 데코레이터를 붙일 수 없는 상황이나 특정 코드 블록만 추적하고 싶을 때 사용해요.

> 🔑 **핵심 개념**: `ls.trace()`의 `rt` 객체는 현재 Run을 나타내요. `rt.end(outputs=...)`로 출력값을 명시적으로 지정할 수 있어요. 지정하지 않으면 `with` 블록이 끝날 때 자동으로 처리돼요.

> ⚠️ **자주 하는 실수**: `with ls.trace(...) as rt:` 구문에서 `rt.end(outputs=...)`를 빠뜨리면 출력 정보가 LangSmith에 기록되지 않을 수 있어요. 중요한 결과값은 명시적으로 `rt.end()`를 호출하는 것이 좋아요.

In [5]:
# ---------------------------------------------------
# ls.trace() 컨텍스트 매니저 사용하기
# ---------------------------------------------------
import langsmith as ls
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def translate_with_trace(text: str, target_language: str) -> str:
    """번역 작업을 수행하고 Trace로 기록해요"""
    # ls.trace(): 코드 블록을 Trace로 감싸요
    # name: Run 이름, run_type: Run 유형, inputs: 입력 정보
    with ls.trace(
        name="번역 파이프라인",
        run_type="chain",
        inputs={"text": text, "target": target_language}
    ) as rt:
        # 이 블록 안의 모든 코드가 하나의 Run으로 추적돼요
        prompt = f"{text}를 {target_language}로 번역해줘. 번역문만 출력해."
        response = model.invoke([HumanMessage(prompt)])
        translated = response.content
        
        # rt.end(): 출력값을 명시적으로 기록해요
        # 지정하지 않으면 반환값이 자동으로 기록돼요
        rt.end(outputs={"translated": translated, "original": text})
        
        return translated

# 번역 실행
result = translate_with_trace("안녕하세요, 반갑습니다!", "영어")
print(f"번역 결과: {result}")

번역 결과: Hello, nice to meet you!


## 5. wrap_openai()로 OpenAI 클라이언트 추적

LangChain을 사용하지 않고 순수 OpenAI SDK를 직접 사용하는 코드도 `wrap_openai()`로 감싸면 LangSmith에 추적할 수 있어요.

> 💡 **실무 팁**: 기존 코드베이스가 OpenAI SDK를 직접 사용하고 있다면, `wrap_openai()`를 적용하기만 해도 LangSmith 추적을 시작할 수 있어요. 클라이언트 초기화 한 줄만 바꾸면 돼요.

In [6]:
# ---------------------------------------------------
# wrap_openai()로 OpenAI 클라이언트 추적하기
# ---------------------------------------------------
import openai
import langsmith as ls
from langsmith.wrappers import wrap_openai

# 기존 OpenAI 클라이언트를 wrap_openai()로 감싸요
# 이후 이 클라이언트를 통한 모든 API 호출이 자동으로 추적돼요
openai_client = wrap_openai(openai.Client())

# 추적될 도구 함수 정의
@ls.traceable(run_type="tool", name="날씨 조회")
def get_weather_info(city: str) -> str:
    """도시의 날씨 정보를 반환해요 (예시: 실제 API 대신 더미 데이터)"""
    # 실제 서비스에서는 날씨 API를 호출하는 부분이에요
    weather_db = {
        "서울": "맑음, 22°C",
        "부산": "구름 많음, 19°C",
        "제주": "비, 18°C"
    }
    return weather_db.get(city, "날씨 정보를 찾을 수 없어요.")

# 전체 파이프라인을 ls.trace()로 감싸요
def weather_chat_pipeline(question: str) -> str:
    """날씨 관련 질문에 답변하는 파이프라인이에요"""
    # 앱 전체 실행을 하나의 Trace로 묶어요
    with ls.trace(
        name="날씨 챗봇",
        run_type="chain",
        inputs={"question": question}
    ) as rt:
        # 도구 함수 호출 (자식 Run으로 추적)
        # 질문에서 도시 이름 추출 (간단한 버전)
        cities = ["서울", "부산", "제주"]
        found_city = None
        for city in cities:
            if city in question:
                found_city = city
                break
        
        weather = get_weather_info(found_city or "서울")  # 자식 Run
        
        # OpenAI SDK로 직접 LLM 호출 (wrap_openai 덕분에 자동 추적)
        completion = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "친절한 날씨 안내 서비스예요."},
                {"role": "user", "content": f"질문: {question}\n날씨 정보: {weather}"}
            ],
            temperature=0
        )
        
        answer = completion.choices[0].message.content
        rt.end(outputs={"answer": answer})
        return answer

# 파이프라인 실행
response = weather_chat_pipeline("서울 날씨 어때요?")
print(f"답변: {response}")

답변: 서울의 날씨는 맑고 기온은 22°C입니다. 외출하기에 좋은 날씨네요! 필요한 추가 정보가 있으면 말씀해 주세요.


## 6. LangSmith UI 탐색 가이드

지금까지 실행한 코드들이 LangSmith에 기록됐어요. UI를 탐색해볼게요.

### Trace 뷰어 화면 구성

```mermaid
flowchart LR
    A[Projects 목록]:::process --> B[Project 상세]
    B --> C[Trace 목록<br/>시간순 정렬]:::output
    C --> D[Trace 상세 뷰]
    D --> E[Run Tree<br/>계층 구조]:::storage
    D --> F[입력값<br/>Input]:::input
    D --> G[출력값<br/>Output]:::output
    D --> H[메타데이터<br/>토큰·비용·시간]:::process

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### UI에서 확인할 수 있는 정보

| 위치 | 확인 내용 |
|------|----------|
| **Run Tree** | 부모-자식 Run 계층 구조와 각 단계의 소요 시간 |
| **Input 탭** | 해당 Run에 전달된 입력값 (프롬프트 전문 포함) |
| **Output 탭** | Run의 출력값 (LLM 응답 전문 포함) |
| **Metadata 탭** | 토큰 수, 예상 비용, 실행 시간, 태그, 메타데이터 |
| **Feedback 탭** | 사람이 남긴 품질 평가 (좋아요/싫어요) |

> 🎯 **강의 포인트**: LangSmith의 진짜 가치는 "왜 이 응답이 나왔는지" 역추적할 때 드러나요. 잘못된 응답을 발견했을 때 Run Tree를 거슬러 올라가며 어느 단계에서 문제가 생겼는지 찾을 수 있어요. 이것이 프로덕션 디버깅의 핵심입니다.

In [7]:
# ---------------------------------------------------
# get_current_run_tree()로 현재 Run에 정보 추가하기
# ---------------------------------------------------
# @traceable 함수 내부에서 get_current_run_tree()를 호출하면
# 현재 실행 중인 Run 객체에 접근할 수 있어요
import langsmith as ls
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

@ls.traceable(run_type="chain", name="동적 메타데이터 추가 예제")
def process_with_dynamic_metadata(user_input: str) -> str:
    """실행 중에 동적으로 메타데이터를 추가하는 예제예요"""
    # 현재 실행 중인 Run 객체를 가져와요
    rt = ls.get_current_run_tree()
    
    # 실행 중에 메타데이터를 동적으로 추가할 수 있어요
    if rt:
        rt.metadata["input_length"] = len(user_input)         # 입력 길이 기록
        rt.metadata["has_question_mark"] = "?" in user_input  # 질문인지 여부
        rt.tags.extend(["동적-태그"])                          # 런타임 태그 추가
    
    # LLM 호출
    response = model.invoke([HumanMessage(user_input)])
    
    # 결과 처리 후 추가 메타데이터
    if rt:
        rt.metadata["output_length"] = len(response.content)
    
    return response.content

# 실행
result = process_with_dynamic_metadata("LangSmith에 대해 한 문장으로 설명해줘.")
print(f"결과: {result}")
print("\nLangSmith UI에서 '동적-태그'와 메타데이터를 확인해보세요!")

결과: LangSmith는 언어 모델 개발 및 관련 기술을 지원하는 플랫폼으로, 자연어 처리 및 AI 시스템의 효율성을 향상시키기 위한 도구와 리소스를 제공합니다.

LangSmith UI에서 '동적-태그'와 메타데이터를 확인해보세요!


In [8]:
# ---------------------------------------------------
# 완전한 파이프라인 예제: @traceable 활용 종합
# ---------------------------------------------------
# ============================================================
# TODO: 아래 파이프라인을 수정해서 새로운 run_type이나 name을 적용해봐요
# 힌트: @traceable(run_type="tool", name="나만의 이름")으로 변경해보세요
# 예상 결과: LangSmith UI에서 지정한 이름으로 Run이 표시돼요
# ============================================================
import langsmith as ls
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

@ls.traceable(run_type="tool", name="키워드 추출")
def extract_keywords(text: str) -> list[str]:
    """텍스트에서 핵심 키워드를 추출해요 (간단한 예시)"""
    # 실제로는 NLP 라이브러리를 사용하지만 여기서는 단순 분할
    words = text.split()
    # 4글자 이상인 단어를 키워드로 간주 (예시)
    keywords = [w for w in words if len(w) >= 4][:3]
    return keywords

@ls.traceable(run_type="llm", name="이메일 초안 작성")
def draft_email(topic: str, keywords: list[str]) -> str:
    """주제와 키워드를 바탕으로 이메일 초안을 작성해요"""
    keyword_str = ", ".join(keywords) if keywords else topic
    messages = [
        SystemMessage("간결하고 전문적인 이메일 초안을 작성해요. 50단어 이내로 써요."),
        HumanMessage(f"주제: {topic}\n핵심 키워드: {keyword_str}")
    ]
    response = model.invoke(messages)
    return response.content

@ls.traceable(run_type="chain", name="이메일 생성 워크플로우")
def email_generation_workflow(raw_input: str) -> dict:
    """원시 입력에서 이메일 초안을 생성하는 전체 워크플로우예요"""
    # 1단계: 키워드 추출
    keywords = extract_keywords(raw_input)
    # 2단계: 이메일 초안 작성
    email_draft = draft_email(raw_input, keywords)
    return {
        "input": raw_input,
        "keywords": keywords,
        "email_draft": email_draft
    }

# 워크플로우 실행
output = email_generation_workflow(
    "다음 주 금요일 오전 10시에 프로젝트 진행 상황 회의 일정 확인 요청"
)
print(f"추출된 키워드: {output['keywords']}")
print(f"\n이메일 초안:\n{output['email_draft']}")

추출된 키워드: ['10시에', '프로젝트']

이메일 초안:
제목: 프로젝트 진행 상황 회의 일정 확인 요청

안녕하세요,

다음 주 금요일 오전 10시에 예정된 프로젝트 진행 상황 회의 일정 확인 부탁드립니다. 참석 여부를 알려주시면 감사하겠습니다.

감사합니다.  
[귀하의 이름]


In [9]:
# ---------------------------------------------------
# 노트북 종료 전 Trace 전송 보장
# ---------------------------------------------------
# Jupyter 노트북에서는 비동기로 Trace를 전송해요
# 노트북 커널을 종료하기 전에 이 셀을 실행해야 해요
from langchain_core.tracers.langchain import wait_for_all_tracers

# 모든 대기 중인 Trace가 LangSmith 서버로 전송될 때까지 기다려요
wait_for_all_tracers()
print("모든 Trace가 LangSmith 서버로 전송됐어요!")
print("smith.langchain.com에서 오늘 실행한 Trace들을 확인해보세요.")

모든 Trace가 LangSmith 서버로 전송됐어요!
smith.langchain.com에서 오늘 실행한 Trace들을 확인해보세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **@traceable**: 일반 Python 함수에 붙여서 LangSmith에 추적해요. `run_type`, `name`, `tags`, `metadata` 파라미터로 세부 설정을 할 수 있어요
- **중첩 Run Tree**: `@traceable` 함수 안에서 다른 `@traceable` 함수를 호출하면 부모-자식 계층 구조가 자동으로 만들어져요
- **langsmith_extra**: 함수 호출 시 런타임에 태그와 메타데이터를 동적으로 추가할 수 있어요
- **ls.trace()**: `with` 블록으로 코드 범위를 직접 지정해 추적할 수 있어요
- **wrap_openai()**: LangChain 없이 순수 OpenAI SDK를 사용하는 코드도 추적할 수 있어요
- **wait_for_all_tracers()**: 노트북에서 모든 Trace가 전송됐는지 보장해요

## 다음 노트북 예고

다음 `03-Chat-Model-Tracing.ipynb`에서는 **ChatOpenAI 호출의 상세한 추적**을 알아봐요. `invoke`, `stream`, `batch`의 Trace 차이, 토큰 수와 비용 추적, 여러 모델의 응답 시간 비교까지 다뤄볼게요.